# Sequence Experiments

Ce notebook reprend le script `sequence_experiments.py`.

But : rendre l'experience lisible et executable dans Jupyter, avec des explications simples en francais.

Utilisation : executez les cellules dans l'ordre. La derniere cellule lance le script avec des arguments controles.


## Pertinence pour le live streaming

Decision : garde. Modele sequence causal: utilise une fenetre d'historique jusqu'a t pour predire le danger court horizon.

Regle appliquee : l'experience doit aider a entrainer, choisir, calibrer, tester ou executer une prediction en flux video avec seulement les informations disponibles a l'instant courant.


## Pourquoi ce notebook est garde

- Description du script : Train real sequence models for danger prediction.
- Commande de reproduction referencee : 15-frame sequence catalogue, 30-frame sequence catalogue, 60-frame sequence catalogue, 60-frame focal catalogue, 60-frame smoothed BCE catalogue.
- Artefacts controles : 15-frame sequence catalogue exists. (`runs/exp_009_sequence_len15_catalogue/metrics/sequence_architecture_comparison.csv`); 30-frame sequence catalogue exists. (`runs/exp_007_sequence_catalogue/metrics/sequence_architecture_comparison.csv`); 60-frame sequence catalogue exists. (`runs/exp_008_sequence_len60_catalogue/metrics/sequence_architecture_comparison.csv`); Focal loss sequence catalogue exists. (`runs/exp_010_sequence_len60_focal_catalogue/metrics/sequence_architecture_comparison.csv`); Label-smoothed sequence catalogue exists. (`runs/exp_011_sequence_len60_smooth_catalogue/metrics/sequence_architecture_comparison.csv`).
- Run par defaut : `runs/exp_007_sequence_catalogue`.

Decision : garde, car il correspond a un artefact experimental, une commande de reproduction, ou un audit du catalogue.


## Avant de commencer

- Verifiez que les donnees et les dossiers `runs/` attendus existent.
- Le notebook n'a pas ete execute pendant sa creation.
- Les cellules de lancement creent un nom ou un dossier unique quand cela evite d'ecraser un resultat existant.


In [ ]:
# Compatibilite Jupyter
# Certains scripts utilisent __file__. Dans un notebook, on le definit explicitement.
from pathlib import Path

PROJECT_ROOT = Path.cwd()
__file__ = str(PROJECT_ROOT / "sequence_experiments.py")


## Importations et configuration

Cette cellule charge les bibliotheques et definit les constantes utilisees par le script.

In [ ]:
import argparse
import json
import math
import random
import time
from collections import Counter
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path

import cv2
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, roc_auc_score
from torch.utils.data import DataLoader, Dataset

from ml_pipeline import (
    HORIZONS,
    ROOT,
    RUNS_DIR,
    TRACKED_PARTS,
    alarm_episodes,
    draw_overlay,
    load_dataset,
    load_json,
    read_frame,
    safe_auc,
    threshold_sweep,
    write_json,
    zone_polygon,
)


FRAME_BASE_COLS = ["person_conf", "bbox_area_norm", "max_signed_dist_norm", "max_signed_dist_vel", "max_signed_dist_acc", "any_part_inside"]


## Fonction `set_seed`

Cette cellule definit `set_seed`. Elle prepare une partie du script.

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


## Fonction `make_run_dir`

Cette cellule definit `make_run_dir`. Elle prepare une partie du script.

In [ ]:
def make_run_dir(run_name):
    run_dir = RUNS_DIR / run_name
    if run_dir.exists():
        raise SystemExit(f"Run directory already exists: {run_dir}")
    for sub in ["features", "models", "metrics", "error_review"]:
        (run_dir / sub).mkdir(parents=True, exist_ok=True)
    return run_dir


## Fonction `append_report`

Cette cellule definit `append_report`. Elle prepare une partie du script.

In [ ]:
def append_report(run_dir, title, body):
    report_path = run_dir / "report.md"
    if not report_path.exists():
        report_path.write_text("# Sequence Model Catalogue\n", encoding="utf-8")
    with report_path.open("a", encoding="utf-8") as handle:
        handle.write(f"\n\n## {title}\n\n")
        handle.write(f"_Updated: {datetime.now().isoformat(timespec='seconds')}_\n\n")
        handle.write(body.rstrip() + "\n")


## Fonction `add_frame_motion_features`

Cette cellule definit `add_frame_motion_features`. Elle prepare une partie du script.

In [ ]:
def add_frame_motion_features(pose):
    rows = []
    for _, group in pose.groupby("video_id", sort=False):
        group = group.sort_values("frame").copy()
        fps = float(group["fps"].iloc[0]) if len(group) else 30.0
        group["max_signed_dist_vel"] = group["max_signed_dist_norm"].diff().fillna(0.0) * fps
        group["max_signed_dist_acc"] = group["max_signed_dist_vel"].diff().fillna(0.0) * fps
        for part in TRACKED_PARTS:
            for axis in ["x", "y"]:
                source = f"{part}_{axis}_norm"
                group[f"{part}_{axis}_vel"] = group[source].diff().fillna(0.0) * fps
            source = f"{part}_signed_dist_norm"
            group[f"{part}_signed_dist_vel"] = group[source].diff().fillna(0.0) * fps
        rows.append(group)
    return pd.concat(rows, ignore_index=True)


## Fonction `frame_feature_columns`

Cette cellule definit `frame_feature_columns`. Elle prepare une partie du script.

In [ ]:
def frame_feature_columns(pose):
    cols = [col for col in FRAME_BASE_COLS if col in pose.columns]
    for part in TRACKED_PARTS:
        cols.extend(
            [
                f"{part}_x_norm",
                f"{part}_y_norm",
                f"{part}_conf",
                f"{part}_signed_dist_norm",
                f"{part}_inside",
                f"{part}_x_vel",
                f"{part}_y_vel",
                f"{part}_signed_dist_vel",
            ]
        )
    return [col for col in cols if col in pose.columns]


## Fonction `build_sequence_dataset`

Cette cellule definit `build_sequence_dataset`. Elle prepare une partie du script.

In [ ]:
def build_sequence_dataset(base_run, run_dir, seq_len):
    base = Path(base_run)
    if not base.is_absolute():
        base = ROOT / base
    pose = pd.read_csv(base / "features" / "pose_features.csv")
    windows = pd.read_csv(base / "features" / "window_features.csv")
    pose = add_frame_motion_features(pose)
    feature_cols = frame_feature_columns(pose)
    pose[feature_cols] = pose[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)

    pose_by_video = {}
    for video_id, group in pose.groupby("video_id", sort=False):
        group = group.sort_values("frame").reset_index(drop=True)
        frame_to_pos = {int(frame): idx for idx, frame in enumerate(group["frame"].tolist())}
        pose_by_video[video_id] = (group, frame_to_pos)

    X = np.zeros((len(windows), seq_len, len(feature_cols)), dtype=np.float32)
    y = np.zeros((len(windows), len(HORIZONS)), dtype=np.float32)
    meta_rows = []
    dropped = 0
    for out_idx, row in windows.reset_index(drop=True).iterrows():
        video_id = row["video_id"]
        frame = int(row["frame"])
        if video_id not in pose_by_video:
            dropped += 1
            continue
        group, frame_to_pos = pose_by_video[video_id]
        end_pos = frame_to_pos.get(frame)
        if end_pos is None:
            end_pos = int(np.argmin(np.abs(group["frame"].to_numpy(dtype=np.int64) - frame)))
        start_pos = max(0, end_pos - seq_len + 1)
        seq = group.iloc[start_pos : end_pos + 1][feature_cols].to_numpy(dtype=np.float32)
        if len(seq) < seq_len:
            pad = np.repeat(seq[:1], seq_len - len(seq), axis=0) if len(seq) else np.zeros((seq_len, len(feature_cols)), dtype=np.float32)
            seq = np.vstack([pad, seq])
        X[out_idx] = seq[-seq_len:]
        for h_idx, horizon in enumerate(HORIZONS):
            y[out_idx, h_idx] = float(row[f"danger_within_{horizon:.1f}s"])
        meta_rows.append(
            {
                "row_idx": out_idx,
                "video_id": video_id,
                "split": row["split"],
                "frame": frame,
                "time_s": float(row["time_s"]),
                "is_danger_clip": int(row["is_danger_clip"]),
                "target_time_s": row.get("target_time_s", ""),
                "time_to_target_s": row.get("time_to_target_s", ""),
                **{f"danger_within_{h:.1f}s": int(row[f"danger_within_{h:.1f}s"]) for h in HORIZONS},
            }
        )

    meta = pd.DataFrame(meta_rows)
    if len(meta) != len(windows):
        X = X[: len(meta)]
        y = y[: len(meta)]
    train_mask = meta["split"].to_numpy() == "train"
    train_flat = X[train_mask].reshape(-1, X.shape[-1])
    mean = train_flat.mean(axis=0)
    std = train_flat.std(axis=0)
    std = np.where(std > 1e-6, std, 1.0)
    X = ((X - mean) / std).astype(np.float32)

    features_dir = run_dir / "features"
    np.savez_compressed(features_dir / "sequence_dataset.npz", X=X, y=y, mean=mean, std=std)
    meta.to_csv(features_dir / "sequence_index.csv", index=False)
    write_json(features_dir / "sequence_feature_columns.json", {"feature_columns": feature_cols, "horizons": HORIZONS, "seq_len": seq_len})

    audit = {
        "base_run": str(base),
        "rows": int(len(meta)),
        "dropped_rows": int(dropped),
        "seq_len": int(seq_len),
        "feature_count": int(len(feature_cols)),
        "split_counts": dict(Counter(meta["split"])),
        "positive_windows_by_horizon": {f"{h:.1f}s": int(meta[f"danger_within_{h:.1f}s"].sum()) for h in HORIZONS},
        "normalization": "train split mean/std only",
    }
    write_json(run_dir / "metrics" / "sequence_dataset_audit.json", audit)
    append_report(
        run_dir,
        "Sequence Dataset Build",
        "\n".join(
            [
                f"- Base run: `{base}`",
                f"- Rows: `{len(meta)}`",
                f"- Sequence length: `{seq_len}` frames",
                f"- Frame features: `{len(feature_cols)}`",
                f"- Split counts: `{audit['split_counts']}`",
                f"- Positive windows: `{audit['positive_windows_by_horizon']}`",
            ]
        ),
    )


## Classe `SequenceDataset`

Cette cellule definit `SequenceDataset`. Elle prepare une partie du script.

In [ ]:
class SequenceDataset(Dataset):
    def __init__(self, X, y, indices, augment=False, seed=42):
        self.X = X
        self.y = y
        self.indices = np.asarray(indices, dtype=np.int64)
        self.augment = augment
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        source_idx = self.indices[idx]
        x = self.X[source_idx].copy()
        y = self.y[source_idx].copy()
        if self.augment:
            x = self.apply_augmentations(x)
        return torch.from_numpy(x), torch.from_numpy(y)

    def apply_augmentations(self, x):
        if self.rng.random() < 0.85:
            x += self.rng.normal(0.0, 0.035, size=x.shape).astype(np.float32)
        if self.rng.random() < 0.35:
            n_features = max(1, int(x.shape[1] * self.rng.uniform(0.03, 0.10)))
            cols = self.rng.choice(x.shape[1], size=n_features, replace=False)
            x[:, cols] = 0.0
        if self.rng.random() < 0.35:
            width = int(self.rng.integers(2, max(3, x.shape[0] // 4)))
            start = int(self.rng.integers(0, max(1, x.shape[0] - width + 1)))
            x[start : start + width] = 0.0
        if self.rng.random() < 0.35:
            scale = float(self.rng.uniform(0.85, 1.18))
            x = resample_sequence_np(x, scale)
        return x.astype(np.float32)


## Fonction `resample_sequence_np`

Cette cellule definit `resample_sequence_np`. Elle prepare une partie du script.

In [ ]:
def resample_sequence_np(x, scale):
    seq_len, feat_dim = x.shape
    source_len = max(4, int(round(seq_len / scale)))
    source_grid = np.linspace(0, seq_len - 1, source_len)
    target_grid = np.linspace(0, source_len - 1, seq_len)
    sampled = np.zeros_like(x)
    for feat in range(feat_dim):
        values = np.interp(source_grid, np.arange(seq_len), x[:, feat])
        sampled[:, feat] = np.interp(target_grid, np.arange(source_len), values)
    return sampled


## Classe `FlattenMLP`

Cette cellule definit `FlattenMLP`. Elle prepare une partie du script.

In [ ]:
class FlattenMLP(nn.Module):
    def __init__(self, seq_len, input_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(seq_len * input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.30),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(128, out_dim),
        )

    def forward(self, x):
        return self.net(x)


## Classe `CNN1D`

Cette cellule definit `CNN1D`. Elle prepare une partie du script.

In [ ]:
class CNN1D(nn.Module):
    def __init__(self, input_dim, out_dim):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(input_dim, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Conv1d(64, 96, kernel_size=3, padding=1),
            nn.BatchNorm1d(96),
            nn.ReLU(),
            nn.Dropout(0.15),
        )
        self.head = nn.Sequential(nn.Linear(96 * 2, 96), nn.ReLU(), nn.Dropout(0.20), nn.Linear(96, out_dim))

    def forward(self, x):
        z = self.conv(x.transpose(1, 2))
        pooled = torch.cat([z.mean(dim=-1), z[:, :, -1]], dim=1)
        return self.head(pooled)


## Classe `TCNBlock`

Cette cellule definit `TCNBlock`. Elle prepare une partie du script.

In [ ]:
class TCNBlock(nn.Module):
    def __init__(self, channels, dilation, dropout):
        super().__init__()
        self.conv1 = nn.Conv1d(channels, channels, kernel_size=3, padding=dilation, dilation=dilation)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size=3, padding=dilation, dilation=dilation)
        self.dropout = nn.Dropout(dropout)
        self.norm1 = nn.BatchNorm1d(channels)
        self.norm2 = nn.BatchNorm1d(channels)

    def forward(self, x):
        residual = x
        x = self.dropout(F.relu(self.norm1(self.conv1(x))))
        x = self.dropout(F.relu(self.norm2(self.conv2(x))))
        return x + residual


## Classe `TCN`

Cette cellule definit `TCN`. Elle prepare une partie du script.

In [ ]:
class TCN(nn.Module):
    def __init__(self, input_dim, out_dim):
        super().__init__()
        self.input = nn.Conv1d(input_dim, 96, kernel_size=1)
        self.blocks = nn.Sequential(*(TCNBlock(96, dilation, 0.18) for dilation in [1, 2, 4, 8]))
        self.head = nn.Sequential(nn.Linear(96 * 2, 96), nn.ReLU(), nn.Dropout(0.20), nn.Linear(96, out_dim))

    def forward(self, x):
        z = self.blocks(self.input(x.transpose(1, 2)))
        pooled = torch.cat([z.mean(dim=-1), z[:, :, -1]], dim=1)
        return self.head(pooled)


## Classe `RecurrentModel`

Cette cellule definit `RecurrentModel`. Elle prepare une partie du script.

In [ ]:
class RecurrentModel(nn.Module):
    def __init__(self, input_dim, out_dim, kind="gru"):
        super().__init__()
        rnn_cls = nn.GRU if kind == "gru" else nn.LSTM
        self.rnn = rnn_cls(input_dim, 96, num_layers=2, batch_first=True, dropout=0.20)
        self.head = nn.Sequential(nn.Linear(96, 96), nn.ReLU(), nn.Dropout(0.20), nn.Linear(96, out_dim))

    def forward(self, x):
        output, _ = self.rnn(x)
        return self.head(output[:, -1])


## Classe `CNNGRU`

Cette cellule definit `CNNGRU`. Elle prepare une partie du script.

In [ ]:
class CNNGRU(nn.Module):
    def __init__(self, input_dim, out_dim):
        super().__init__()
        self.conv = nn.Sequential(nn.Conv1d(input_dim, 64, kernel_size=3, padding=1), nn.ReLU(), nn.Dropout(0.15))
        self.rnn = nn.GRU(64, 96, num_layers=1, batch_first=True)
        self.head = nn.Sequential(nn.Linear(96, 96), nn.ReLU(), nn.Dropout(0.20), nn.Linear(96, out_dim))

    def forward(self, x):
        z = self.conv(x.transpose(1, 2)).transpose(1, 2)
        output, _ = self.rnn(z)
        return self.head(output[:, -1])


## Classe `TemporalTransformer`

Cette cellule definit `TemporalTransformer`. Elle prepare une partie du script.

In [ ]:
class TemporalTransformer(nn.Module):
    def __init__(self, seq_len, input_dim, out_dim):
        super().__init__()
        self.proj = nn.Linear(input_dim, 96)
        self.pos = nn.Parameter(torch.zeros(1, seq_len, 96))
        layer = nn.TransformerEncoderLayer(d_model=96, nhead=4, dim_feedforward=192, dropout=0.20, batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=2)
        self.head = nn.Sequential(nn.Linear(96, 96), nn.ReLU(), nn.Dropout(0.20), nn.Linear(96, out_dim))

    def forward(self, x):
        z = self.proj(x) + self.pos[:, : x.shape[1]]
        z = self.encoder(z)
        return self.head(z[:, -1])


## Fonction `make_model`

Cette cellule definit `make_model`. Elle prepare une partie du script.

In [ ]:
def make_model(kind, seq_len, input_dim, out_dim):
    if kind == "flatten_mlp":
        return FlattenMLP(seq_len, input_dim, out_dim)
    if kind == "cnn1d":
        return CNN1D(input_dim, out_dim)
    if kind == "tcn":
        return TCN(input_dim, out_dim)
    if kind == "gru":
        return RecurrentModel(input_dim, out_dim, "gru")
    if kind == "lstm":
        return RecurrentModel(input_dim, out_dim, "lstm")
    if kind == "cnn_gru":
        return CNNGRU(input_dim, out_dim)
    if kind == "transformer":
        return TemporalTransformer(seq_len, input_dim, out_dim)
    raise ValueError(f"Unknown model kind: {kind}")


## Classe `SmoothedBCEWithLogitsLoss`

Cette cellule definit `SmoothedBCEWithLogitsLoss`. Elle prepare une partie du script.

In [ ]:
class SmoothedBCEWithLogitsLoss(nn.Module):
    def __init__(self, pos_weight, smoothing):
        super().__init__()
        self.register_buffer("pos_weight", pos_weight)
        self.smoothing = float(smoothing)

    def forward(self, logits, targets):
        if self.smoothing > 0:
            targets = targets * (1.0 - self.smoothing) + 0.5 * self.smoothing
        return F.binary_cross_entropy_with_logits(logits, targets, pos_weight=self.pos_weight)


## Classe `FocalBCEWithLogitsLoss`

Cette cellule definit `FocalBCEWithLogitsLoss`. Elle prepare une partie du script.

In [ ]:
class FocalBCEWithLogitsLoss(nn.Module):
    def __init__(self, pos_weight, gamma=2.0):
        super().__init__()
        self.register_buffer("pos_weight", pos_weight)
        self.gamma = float(gamma)

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, pos_weight=self.pos_weight, reduction="none")
        pt = torch.exp(-bce).clamp(min=1e-6, max=1.0)
        return (((1.0 - pt) ** self.gamma) * bce).mean()


## Fonction `make_loss`

Cette cellule definit `make_loss`. Elle prepare une partie du script.

In [ ]:
def make_loss(args, pos_weight):
    if args.loss == "bce":
        return nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    if args.loss == "smooth_bce":
        return SmoothedBCEWithLogitsLoss(pos_weight, args.label_smoothing)
    if args.loss == "focal":
        return FocalBCEWithLogitsLoss(pos_weight, args.focal_gamma)
    raise ValueError(f"Unknown loss: {args.loss}")


## Fonction `predict_model`

Cette cellule definit `predict_model`. Elle prepare une partie du script.

In [ ]:
@torch.no_grad()
def predict_model(model, loader, device):
    model.eval()
    probs = []
    targets = []
    for xb, yb in loader:
        xb = xb.to(device)
        logits = model(xb)
        probs.append(torch.sigmoid(logits).cpu().numpy())
        targets.append(yb.numpy())
    return np.vstack(probs), np.vstack(targets)


## Fonction `train_one_model`

Cette cellule definit `train_one_model`. Elle prepare une partie du script.

In [ ]:
def train_one_model(name, kind, augment, X, y, meta, run_dir, args, device):
    seq_len, input_dim, out_dim = X.shape[1], X.shape[2], y.shape[1]
    train_idx = np.flatnonzero(meta["split"].to_numpy() == "train")
    val_idx = np.flatnonzero(meta["split"].to_numpy() == "val")
    train_ds = SequenceDataset(X, y, train_idx, augment=augment, seed=args.seed)
    val_ds = SequenceDataset(X, y, val_idx, augment=False, seed=args.seed)
    train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=args.batch_size, shuffle=False, num_workers=0)

    model = make_model(kind, seq_len, input_dim, out_dim).to(device)
    positives = y[train_idx].sum(axis=0)
    negatives = len(train_idx) - positives
    pos_weight = torch.tensor(np.clip(negatives / np.maximum(positives, 1.0), 1.0, 20.0), dtype=torch.float32, device=device)
    criterion = make_loss(args, pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)

    best = {"score": -1.0, "epoch": 0, "state": None}
    history = []
    patience_left = args.patience
    start = time.perf_counter()
    for epoch in range(1, args.epochs + 1):
        model.train()
        losses = []
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 3.0)
            optimizer.step()
            losses.append(float(loss.detach().cpu()))
        val_probs, val_targets = predict_model(model, val_loader, device)
        h1_idx = HORIZONS.index(1.0)
        val_ap = safe_auc(average_precision_score, val_targets[:, h1_idx], val_probs[:, h1_idx])
        val_score = float(val_ap or 0.0)
        scheduler.step(val_score)
        row = {"model": name, "epoch": epoch, "train_loss": float(np.mean(losses)), "val_ap_h1": val_score, "lr": float(optimizer.param_groups[0]["lr"])}
        history.append(row)
        if val_score > best["score"] + 1e-5:
            best = {**best, "score": val_score, "epoch": epoch, "state": {k: v.detach().cpu() for k, v in model.state_dict().items()}}
            patience_left = args.patience
        else:
            patience_left -= 1
        if patience_left <= 0:
            break

    if best["state"] is not None:
        model.load_state_dict(best["state"])
    train_time = time.perf_counter() - start
    model_path = run_dir / "models" / f"{name}.pt"
    torch.save(
        {
            "model_name": name,
            "kind": kind,
            "augment": augment,
            "state_dict": model.state_dict(),
            "seq_len": seq_len,
            "input_dim": input_dim,
            "horizons": HORIZONS,
            "best_epoch": best["epoch"],
            "best_val_ap_h1": best["score"],
        },
        model_path,
    )
    return model, history, train_time, model_path.stat().st_size


## Fonction `evaluate_catalogue_model`

Cette cellule definit `evaluate_catalogue_model`. Elle prepare une partie du script.

In [ ]:
def evaluate_catalogue_model(name, model, X, y, meta, run_dir, device, train_time_s, model_size_bytes, batch_size, loss_name):
    all_idx = np.arange(len(meta))
    loader = DataLoader(SequenceDataset(X, y, all_idx, augment=False), batch_size=batch_size, shuffle=False, num_workers=0)
    start = time.perf_counter()
    probs, targets = predict_model(model, loader, device)
    inference_s = time.perf_counter() - start
    pred = meta.copy()
    for h_idx, horizon in enumerate(HORIZONS):
        pred[f"prob_{horizon:.1f}s"] = probs[:, h_idx]
    pred.to_csv(run_dir / "features" / f"predictions_{name}.csv", index=False)

    comparison_rows = []
    sweep_rows = []
    h1_idx = HORIZONS.index(1.0)
    for split in ["train", "val", "test"]:
        split_df = pred[pred["split"] == split].copy()
        for horizon in HORIZONS:
            y_col = f"danger_within_{horizon:.1f}s"
            p_col = f"prob_{horizon:.1f}s"
            y_true = split_df[y_col].astype(int).to_numpy()
            y_score = split_df[p_col].to_numpy()
            row = {
                "architecture": name,
                "loss": loss_name,
                "split": split,
                "horizon_s": horizon,
                "n": int(len(split_df)),
                "positive": int(y_true.sum()),
                "roc_auc": safe_auc(roc_auc_score, y_true, y_score),
                "average_precision": safe_auc(average_precision_score, y_true, y_score),
                "train_time_s": float(train_time_s),
                "inference_ms_per_window": float(inference_s * 1000.0 / max(1, len(pred))),
                "model_size_bytes": int(model_size_bytes),
            }
            if horizon == 1.0:
                sweep = threshold_sweep(split_df.rename(columns={p_col: "risk"}), "risk", 1.0, split, persistence_windows=2)
                sweep["architecture"] = name
                sweep_rows.append(sweep)
                best = sweep.sort_values(["danger_clip_hit_rate", "safe_false_alarms_per_min", "window_precision"], ascending=[False, True, False]).iloc[0]
                row.update(
                    {
                        "best_threshold_by_hit_fa": float(best["threshold"]),
                        "best_hit_rate": float(best["danger_clip_hit_rate"]),
                        "best_false_alarms_per_min": float(best["safe_false_alarms_per_min"]),
                        "best_window_precision": float(best["window_precision"]),
                        "best_window_recall": float(best["window_recall"]),
                    }
                )
            comparison_rows.append(row)
    return comparison_rows, sweep_rows


## Fonction `make_sequence_augmentation_proof`

Cette cellule definit `make_sequence_augmentation_proof`. Elle prepare une partie du script.

In [ ]:
def make_sequence_augmentation_proof(run_dir, X, meta):
    out_dir = run_dir / "error_review" / "sequence_augmentation_examples"
    out_dir.mkdir(parents=True, exist_ok=True)
    train_idx = np.flatnonzero(meta["split"].to_numpy() == "train")[:6]
    if len(train_idx) == 0:
        return
    rng = np.random.default_rng(123)
    tiles = []
    for idx in train_idx:
        original = X[idx]
        augmented = original.copy()
        augmented += rng.normal(0.0, 0.035, size=augmented.shape).astype(np.float32)
        augmented = resample_sequence_np(augmented, 1.12)
        canvas = np.full((220, 360, 3), 255, dtype=np.uint8)
        for series, color, yoff, label in [
            (original[:, 2], (60, 120, 220), 80, "orig distance"),
            (augmented[:, 2], (220, 80, 70), 160, "aug distance"),
        ]:
            vals = np.asarray(series, dtype=np.float32)
            vals = (vals - vals.min()) / max(1e-6, vals.max() - vals.min())
            pts = []
            for i, val in enumerate(vals):
                x = 20 + int(i * (320 / max(1, len(vals) - 1)))
                y = yoff - int((val - 0.5) * 80)
                pts.append((x, y))
            for p1, p2 in zip(pts[:-1], pts[1:]):
                cv2.line(canvas, p1, p2, color, 2)
            cv2.putText(canvas, label, (18, yoff + 35), cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1, cv2.LINE_AA)
        cv2.putText(canvas, str(meta.iloc[idx]["video_id"])[:36], (18, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (40, 40, 40), 1, cv2.LINE_AA)
        tiles.append(canvas)
    sheet = np.vstack(tiles)
    cv2.imwrite(str(out_dir / "sequence_augmentation_distance_contact_sheet.jpg"), sheet)


## Fonction `make_sequence_failure_screenshots`

Cette cellule definit `make_sequence_failure_screenshots`. Elle prepare une partie du script.

In [ ]:
def make_sequence_failure_screenshots(run_dir, best_model_name, threshold):
    pred_path = run_dir / "features" / f"predictions_{best_model_name}.csv"
    if not pred_path.exists():
        return
    pred = pd.read_csv(pred_path)
    out_dir = run_dir / "error_review" / "sequence_failures"
    out_dir.mkdir(parents=True, exist_ok=True)
    videos, _, _, zones = load_dataset()
    video_by_id = {row["video_id"]: row for row in videos}
    polygon = zone_polygon(zones)
    saved = 0
    for video_id, group in pred[pred["split"] == "test"].groupby("video_id"):
        group = group.sort_values("time_s")
        is_danger = int(group["is_danger_clip"].iloc[0])
        target_values = group["target_time_s"].replace("", np.nan).astype(float).dropna()
        target = float(target_values.iloc[0]) if len(target_values) else math.nan
        alarms = alarm_episodes(group["time_s"], group["prob_1.0s"], threshold, gap_s=1.0, persistence_windows=2)
        failure = False
        frame_time = None
        label = ""
        if is_danger:
            useful = [t for t in alarms if not math.isnan(target) and t <= target + 0.5]
            if not useful:
                failure = True
                frame_time = target if not math.isnan(target) else float(group["time_s"].iloc[-1])
                label = "SEQ MISS"
        elif alarms:
            failure = True
            frame_time = min(alarms)
            label = "SEQ FALSE POSITIVE"
        if not failure or saved >= 16 or video_id not in video_by_id:
            continue
        video = video_by_id[video_id]
        frame_idx = int(round(frame_time * float(video["fps"])))
        frame = read_frame(ROOT / video["path"], frame_idx)
        if frame is None:
            continue
        nearest = group.iloc[(group["time_s"] - frame_time).abs().argsort().iloc[0]]
        out = draw_overlay(frame, polygon, [label, f"{best_model_name}={float(nearest['prob_1.0s']):.3f} thr={threshold:.2f}", video["path"]])
        cv2.imwrite(str(out_dir / f"{video_id}_{label.lower().replace(' ', '_')}.jpg"), out)
        saved += 1


## Fonction `summarize_against_tabular`

Cette cellule definit `summarize_against_tabular`. Elle prepare une partie du script.

In [ ]:
def summarize_against_tabular(base_run, run_dir, sequence_metrics):
    base = Path(base_run)
    if not base.is_absolute():
        base = ROOT / base
    lines = ["# Sequence Vs Tabular Summary", ""]
    lines.append("## Sequence Models")
    val = sequence_metrics[(sequence_metrics["split"] == "val") & (sequence_metrics["horizon_s"] == 1.0)].copy()
    val["selection_score"] = (
        val["average_precision"].fillna(0)
        + 0.5 * val["best_hit_rate"].fillna(0)
        + 0.2 * val["best_window_precision"].fillna(0)
        - 0.03 * val["best_false_alarms_per_min"].fillna(20).clip(upper=20)
    )
    best = val.sort_values("selection_score", ascending=False).iloc[0]
    lines.append(f"- Best sequence validation model: `{best['architecture']}`")
    lines.append(f"- Validation AP: `{float(best['average_precision']):.4f}`")
    lines.append(f"- Validation hit rate: `{float(best['best_hit_rate']):.4f}`")
    lines.append(f"- Validation false alarms/min: `{float(best['best_false_alarms_per_min']):.4f}`")
    lines.append("")
    tabular_path = base / "metrics" / "architecture_comparison.csv"
    if tabular_path.exists():
        tab = pd.read_csv(tabular_path)
        tab_val = tab[tab["split"] == "val"].copy()
        tab_val["selection_score"] = (
            tab_val["average_precision"].fillna(0)
            + 0.5 * tab_val["best_hit_rate"].fillna(0)
            + 0.2 * tab_val["best_window_precision"].fillna(0)
            - 0.03 * tab_val["best_false_alarms_per_min"].fillna(20).clip(upper=20)
        )
        tab_best = tab_val.sort_values("selection_score", ascending=False).iloc[0]
        lines.append("## Previous Tabular-Window Baseline")
        lines.append(f"- Best tabular validation model: `{tab_best['architecture']}`")
        lines.append(f"- Validation AP: `{float(tab_best['average_precision']):.4f}`")
        lines.append(f"- Validation hit rate: `{float(tab_best['best_hit_rate']):.4f}`")
        lines.append(f"- Validation false alarms/min: `{float(tab_best['best_false_alarms_per_min']):.4f}`")
    lines.append("")
    lines.append("## Interpretation")
    lines.append("")
    lines.append("- Sequence models use frame-by-frame pose histories, so they are the correct architecture family for the danger prediction problem.")
    lines.append("- Tabular-window models remain useful baselines, but they should not be described as the final temporal model.")
    (run_dir / "sequence_vs_tabular_summary.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
    return best


## Fonction `train_catalogue`

Cette cellule definit `train_catalogue`. Elle prepare une partie du script.

In [ ]:
def train_catalogue(args):
    set_seed(args.seed)
    run_dir = make_run_dir(args.run_name)
    write_json(
        run_dir / "config.json",
        {
            "base_run": args.base_run,
            "seed": args.seed,
            "seq_len": args.seq_len,
            "epochs": args.epochs,
            "patience": args.patience,
            "batch_size": args.batch_size,
            "lr": args.lr,
            "weight_decay": args.weight_decay,
            "loss": args.loss,
            "label_smoothing": args.label_smoothing,
            "focal_gamma": args.focal_gamma,
            "created_at": datetime.now().isoformat(timespec="seconds"),
        },
    )
    build_sequence_dataset(args.base_run, run_dir, args.seq_len)
    data = np.load(run_dir / "features" / "sequence_dataset.npz")
    X = data["X"].astype(np.float32)
    y = data["y"].astype(np.float32)
    meta = pd.read_csv(run_dir / "features" / "sequence_index.csv")
    make_sequence_augmentation_proof(run_dir, X, meta)
    write_json(
        run_dir / "metrics" / "sequence_augmentation_audit.json",
        {
            "train_only": True,
            "stochastic_training_augmentations": [
                "gaussian_feature_jitter",
                "feature_channel_dropout",
                "temporal_cutout",
                "temporal_speed_resample",
            ],
            "proof_contact_sheet": "error_review/sequence_augmentation_examples/sequence_augmentation_distance_contact_sheet.jpg",
            "non_train_augmented": 0,
        },
    )

    if args.device == "auto":
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    else:
        device = torch.device(args.device)
    arch_specs = [
        ("flatten_mlp_aug", "flatten_mlp", True),
        ("cnn1d_aug", "cnn1d", True),
        ("tcn_aug", "tcn", True),
        ("gru_aug", "gru", True),
        ("lstm_aug", "lstm", True),
        ("cnn_gru_aug", "cnn_gru", True),
        ("transformer_aug", "transformer", True),
        ("tcn_noaug", "tcn", False),
        ("gru_noaug", "gru", False),
    ]
    if args.quick:
        arch_specs = arch_specs[:4]

    all_history = []
    all_comparison = []
    all_sweeps = []
    for name, kind, augment in arch_specs:
        print(f"training {name} on {device}")
        model, history, train_time_s, model_size_bytes = train_one_model(name, kind, augment, X, y, meta, run_dir, args, device)
        all_history.extend(history)
        comparison_rows, sweep_rows = evaluate_catalogue_model(name, model, X, y, meta, run_dir, device, train_time_s, model_size_bytes, args.batch_size, args.loss)
        all_comparison.extend(comparison_rows)
        all_sweeps.extend(sweep_rows)
        pd.DataFrame(all_history).to_csv(run_dir / "metrics" / "sequence_training_history.csv", index=False)
        pd.DataFrame(all_comparison).to_csv(run_dir / "metrics" / "sequence_architecture_comparison.csv", index=False)
        if all_sweeps:
            pd.concat(all_sweeps, ignore_index=True).to_csv(run_dir / "metrics" / "sequence_threshold_sweeps.csv", index=False)

    comparison = pd.DataFrame(all_comparison)
    comparison.to_csv(run_dir / "metrics" / "sequence_architecture_comparison.csv", index=False)
    pd.DataFrame(all_history).to_csv(run_dir / "metrics" / "sequence_training_history.csv", index=False)
    if all_sweeps:
        pd.concat(all_sweeps, ignore_index=True).to_csv(run_dir / "metrics" / "sequence_threshold_sweeps.csv", index=False)
    best = summarize_against_tabular(args.base_run, run_dir, comparison)
    make_sequence_failure_screenshots(run_dir, str(best["architecture"]), float(best["best_threshold_by_hit_fa"]))
    append_report(
        run_dir,
        "Sequence Catalogue Completion",
        "\n".join(
            [
                f"- Architectures trained: `{[name for name, _, _ in arch_specs]}`",
                f"- Best validation sequence model: `{best['architecture']}`",
                f"- Best validation AP: `{float(best['average_precision']):.4f}`",
                f"- Outputs: `{run_dir}`",
            ]
        ),
    )
    print(run_dir)


## Point d'entree principal

Cette cellule definit `main`. Elle prepare une partie du script.

In [ ]:
def main():
    parser = argparse.ArgumentParser(description="Train real sequence models for danger prediction.")
    parser.add_argument("--base-run", default="runs/exp_006_final_research")
    parser.add_argument("--run-name", default="exp_007_sequence_catalogue")
    parser.add_argument("--seq-len", type=int, default=30)
    parser.add_argument("--epochs", type=int, default=35)
    parser.add_argument("--patience", type=int, default=7)
    parser.add_argument("--batch-size", type=int, default=128)
    parser.add_argument("--lr", type=float, default=1e-3)
    parser.add_argument("--weight-decay", type=float, default=1e-4)
    parser.add_argument("--loss", choices=["bce", "smooth_bce", "focal"], default="bce")
    parser.add_argument("--label-smoothing", type=float, default=0.05)
    parser.add_argument("--focal-gamma", type=float, default=2.0)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--device", default="auto")
    parser.add_argument("--quick", action="store_true")
    args = parser.parse_args()
    train_catalogue(args)


## Lancer le script

Cette cellule lance le `main()` avec des arguments adaptes au notebook.

In [ ]:
# Lancement du script
# Modifiez NOTEBOOK_ARGS si vous voulez changer les options.
from datetime import datetime
import sys

RUN_NAME_BASE = "exp_007_sequence_catalogue_notebook"
RUN_NAME = f"{RUN_NAME_BASE}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
NOTEBOOK_ARGS = ["--run-name", RUN_NAME]

ancien_argv = sys.argv[:]
sys.argv = ["sequence_experiments.py"] + NOTEBOOK_ARGS
try:
    print("Arguments utilises :", sys.argv[1:])
    main()
finally:
    sys.argv = ancien_argv
